# Fine-tune NTv3 on crop stress & disease-resistance genes

This notebook runs the Nucleotide Transformer v3 pipeline on **Google Colab**:

1. Attach a GPU
2. Load this repo
3. Fine-tune NTv3 as a multi-label gene classifier
4. Rank candidate genes for drought, salt, heat, cold, flooding, pathogen, and insect resistance

| GPU | Config | Model |
|---|---|---|
| T4 16 GB (Colab free) | `configs/colab_t4.yaml` | `NTv3_100M_pre` + LoRA |
| A100 40 GB (Colab Pro) | `configs/colab_a100.yaml` | `NTv3_650M_pre` + LoRA |
| CPU / no GPU | `configs/mac_8m.yaml` | `NTv3_8M_pre` smoke test |

**Before you run anything:** `Runtime → Change runtime type → T4 GPU` (or A100 if you have Colab Pro).



## 1. Check the GPU



In [ ]:
import torch

print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(torch.cuda.get_device_name(0))
    print(f"VRAM {props.total_memory / 1024**3:.1f} GB")
else:
    print("No GPU. Runtime → Change runtime type → T4 GPU, then rerun from the top.")



In [ ]:
!nvidia-smi -L || true



## 2. Get this repo onto Colab

Pick one:

- **upload** — zip the project on your Mac, then upload it in the next cell
- **drive** — put the `NucleotideTransformer` folder in Google Drive
- **github** — paste a public clone URL (only if you have pushed this repo)



In [ ]:
# @title Repo source
SOURCE = "upload"  # @param ["upload", "drive", "github"]
GITHUB_URL = "https://github.com/YOUR_USER/NucleotideTransformer.git"  # @param {type:"string"}
DRIVE_PATH = "/content/drive/MyDrive/NucleotideTransformer"  # @param {type:"string"}



In [ ]:
from pathlib import Path
import os, zipfile, sys, subprocess

def already_here(start=Path(".")):
    start = Path(start)
    candidates = [start, start / "NucleotideTransformer", Path("/content/NucleotideTransformer")]
    for cand in candidates:
        if (cand / "src" / "ntv3_crop").exists():
            return cand.resolve()
    matches = list(start.rglob("src/ntv3_crop"))
    if matches:
        return matches[0].parent.parent.resolve()
    return None

ROOT = already_here()

if ROOT is None and SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = already_here(Path(DRIVE_PATH)) or Path(DRIVE_PATH).resolve()

elif ROOT is None and SOURCE == "github":
    if "YOUR_USER" in GITHUB_URL:
        raise SystemExit("Set GITHUB_URL to your clone URL, or switch SOURCE to upload/drive.")
    subprocess.check_call(["git", "clone", "--depth", "1", GITHUB_URL, "/content/NucleotideTransformer"])
    ROOT = Path("/content/NucleotideTransformer")

elif ROOT is None and SOURCE == "upload":
    from google.colab import files
    print("Upload a zip of the NucleotideTransformer folder.")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file uploaded.")
    name = next(iter(uploaded))
    dest = Path("/content/uploaded_repo")
    dest.mkdir(exist_ok=True)
    with zipfile.ZipFile(name) as zf:
        zf.extractall(dest)
    ROOT = already_here(dest)
    if ROOT is None:
        raise SystemExit("Zip extracted but src/ntv3_crop was not found. Zip the project folder itself.")

if ROOT is None or not (ROOT / "src" / "ntv3_crop").exists():
    raise SystemExit(f"Could not find the repo at {ROOT}")

os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)



On your Mac, create the zip with:

```bash
cd /Users/mac
zip -r NucleotideTransformer.zip NucleotideTransformer \
  -x "NucleotideTransformer/.venv/*" \
  -x "NucleotideTransformer/outputs/*"
```



## 3. Install packages



In [ ]:
%pip -q install -U pip
%pip -q install -U "transformers>=4.55.0" "accelerate>=0.33.0" "peft>=0.12.0" \
  "datasets>=2.20.0" "biopython>=1.83" "scikit-learn>=1.4" pyyaml pandas tqdm evaluate
%pip -q install -e .



## 4. Hugging Face login

Some NTv3 files are gated. Create a token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) with **read** access, then accept the model terms if prompted:
[NTv3_100M_pre](https://huggingface.co/InstaDeepAI/NTv3_100M_pre) · [NTv3_650M_pre](https://huggingface.co/InstaDeepAI/NTv3_650M_pre)



In [ ]:
from huggingface_hub import notebook_login
notebook_login()



## 5. Choose data and training length

Leave **sample** on for the first run. It uses planted DNA motifs so you can confirm training works in a few minutes. Switch to **ensembl** for real Arabidopsis / rice / maize GO-labeled genes (slow: Ensembl REST downloads).



In [ ]:
# @title Training options
DATA_MODE = "sample"  # @param ["sample", "ensembl"]
SPECIES = "arabidopsis_thaliana,oryza_sativa"  # @param {type:"string"}
MAX_GENES_PER_SPECIES = 400  # @param {type:"integer"}
SMOKE_MAX_STEPS = 80  # @param {type:"integer"}
OVERRIDE_CONFIG = ""  # @param {type:"string"}



In [ ]:
from ntv3_crop.hardware import recommend_config
from ntv3_crop.config import load_config

config_path = OVERRIDE_CONFIG.strip() or recommend_config()
cfg = load_config(config_path)
print("Using", config_path)
print("model", cfg.model_id, "| seq_len", cfg.seq_len, "| lora", cfg.use_lora)
print("fp16", cfg.fp16, "bf16", cfg.bf16)



In [ ]:
from pathlib import Path
import subprocess, sys

if DATA_MODE == "sample":
    subprocess.check_call([sys.executable, "scripts/prepare_sample.py", "--seq-len", str(cfg.seq_len)])
    cfg.train_csv = "data/sample/train.csv"
    cfg.val_csv = "data/sample/val.csv"
    cfg.output_dir = "outputs/colab_sample"
    if SMOKE_MAX_STEPS and SMOKE_MAX_STEPS > 0:
        cfg.max_steps = int(SMOKE_MAX_STEPS)
        cfg.eval_steps = min(cfg.eval_steps, max(10, cfg.max_steps // 2))
        cfg.save_steps = cfg.eval_steps
    print("Sample data ready (synthetic motifs, not biology).")
else:
    species = [s.strip() for s in SPECIES.split(",") if s.strip()]
    cmd = [
        sys.executable, "scripts/prepare_ensembl.py",
        "--species", *species,
        "--out-dir", "data/processed",
        "--max-genes", str(MAX_GENES_PER_SPECIES),
    ]
    subprocess.check_call(cmd)
    cfg.train_csv = "data/processed/train.csv"
    cfg.val_csv = "data/processed/val.csv"
    cfg.output_dir = "outputs/colab_ensembl"
    cfg.max_steps = -1
    print("Ensembl GO-labeled data ready.")

print("train", cfg.train_csv, "→", cfg.output_dir)



## 6. Fine-tune



In [ ]:
from ntv3_crop.train import train

best_dir = train(cfg)
print("Best checkpoint:", best_dir)



## 7. Rank candidate genes



In [ ]:
from ntv3_crop.discover import load_trained, score_sequences
from Bio import SeqIO
from pathlib import Path

fasta = Path("data/sample/candidates.fa")
if DATA_MODE == "ensembl" and Path("data/processed/val.csv").exists():
    # Score held-out validation genes written as FASTA on the fly.
    import pandas as pd
    from Bio.Seq import Seq
    from Bio.SeqRecord import SeqRecord
    val = pd.read_csv(cfg.val_csv)
    fasta = Path("data/processed/val_candidates.fa")
    recs = [SeqRecord(Seq(str(r.sequence)), id=str(r.gene_id), description="") for r in val.itertuples()]
    SeqIO.write(recs, fasta, "fasta")

model, tokenizer, labels = load_trained(best_dir, cfg)
records = [(rec.id, str(rec.seq)) for rec in SeqIO.parse(fasta, "fasta")]
table = score_sequences(model, tokenizer, records, labels, cfg.seq_len, batch_size=2)
out = Path(cfg.output_dir) / "discoveries.tsv"
out.parent.mkdir(parents=True, exist_ok=True)
table.to_csv(out, sep="\t", index=False)
print(table.head(20).to_string(index=False))
print("\nWrote", out)



## 8. Save results

Download the ranked gene table and checkpoint, or copy them to Drive so a Colab disconnect does not wipe them.



In [ ]:
from pathlib import Path
from google.colab import files
import shutil

out_tsv = Path(cfg.output_dir) / "discoveries.tsv"
if out_tsv.exists():
    files.download(str(out_tsv))

save_to_drive = True  # @param {type:"boolean"}
if save_to_drive:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    dest = Path("/content/drive/MyDrive/NTv3_crop_outputs")
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copytree(cfg.output_dir, dest / Path(cfg.output_dir).name, dirs_exist_ok=True)
    print("Copied", cfg.output_dir, "→", dest)



## Next

- First run should use **sample** data. If loss drops, the stack works.
- Then set `DATA_MODE` to **ensembl** and rerun from the data cell.
- On T4, stay on `NTv3_100M_pre`. Only switch to 650M on an A100.
- If you OOM, lower `cfg.seq_len` to `1024` (must stay a multiple of 128) and rerun training.

